In [9]:
import os
import glob
import pandas as pd
import geopandas as gpd

# Resolve base data directory whether executed from root or notebooks/
data_dir = "data" if os.path.exists("data") else "../data"

state_files = [
    {"name": "alabama2026.geojson", "postal": "AL"},
    {"name": "california2026.geojson", "postal": "CA"},
    {"name": "florida2026.geojson", "postal": "FL"},
    {"name": "louisiana2026.geojson", "postal": "LA"},
    {"name": "northcarolina2026.geojson", "postal": "NC"},
    {"name": "ohio2026.geojson", "postal": "OH"},
    {"name": "tennessee2026.geojson", "postal": "TN"},
    {"name": "texas2026.geojson", "postal": "TX"},
    {"name": "utah2026.geojson", "postal": "UT"},
]

TARGET_CRS = "EPSG:4326"
all_gdfs = []

# Standardize projections and district ID format
for item in state_files:
    file_path = os.path.join(data_dir, item["name"])
    gdf = gpd.read_file(file_path)

    if gdf.crs != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)
        
    gdf["District"] = item["postal"] + gdf["id"].astype(str).str.zfill(2)
    
    gdf = gdf[[
        "id", "District", "DemPct", "RepPct", "WhitePct", "MinorityPct", 
        "BlackPct", "HispanicPct", "AsianPct", "Margin", "geometry"
    ]]
    
    all_gdfs.append(gdf)

states_gdf = pd.concat(all_gdfs, ignore_index=True)
states_gdf.head(8)

,id,District,DemPct,RepPct,WhitePct,MinorityPct,BlackPct,HispanicPct,AsianPct,Margin,geometry
0,1,AL01,0.314327,0.673261,0.670796,0.329204,0.246268,0.032630,0.018676,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ..."
1,2,AL02,0.422827,0.565831,0.524975,0.475025,0.399287,0.037441,0.022071,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ..."
2,3,AL03,0.261882,0.727057,0.717585,0.282415,0.206950,0.031171,0.021240,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ..."
3,4,AL04,0.163376,0.826469,0.828291,0.171709,0.072246,0.060967,0.008272,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ..."
4,5,AL05,0.346103,0.635919,0.702694,0.297306,0.183322,0.052956,0.025062,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ..."
5,6,AL06,0.320528,0.662526,0.716493,0.283507,0.192588,0.046781,0.021853,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ..."
6,7,AL07,0.582883,0.405147,0.427028,0.572972,0.506518,0.036561,0.016651,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ..."
7,1,CA01,0.544906,0.422732,0.708434,0.291566,0.023875,0.191664,0.040873,0.044906,"MULTIPOLYGON (((-121.32264 41.18387, -121.3308..."


In [10]:
# Load 2024 presidential totals from old boundaries
pres24_path = os.path.join(data_dir, "pres2024", "2024_pres.csv")
pres24 = pd.read_csv(pres24_path)[['District', 'Harris', 'Trump 24', 'Total']]

merged_gdf = states_gdf.merge(pres24, on='District', how='inner')
merged_gdf.head(8)

,id,District,DemPct,RepPct,WhitePct,MinorityPct,BlackPct,HispanicPct,AsianPct,Margin,geometry,Harris,Trump 24,Total
0,1,AL01,0.314327,0.673261,0.670796,0.329204,0.246268,0.032630,0.018676,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",73003,257060,332700
1,2,AL02,0.422827,0.565831,0.524975,0.475025,0.399287,0.037441,0.022071,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",155603,131721,290033
2,3,AL03,0.261882,0.727057,0.717585,0.282415,0.206950,0.031171,0.021240,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",82654,229676,314869
3,4,AL04,0.163376,0.826469,0.828291,0.171709,0.072246,0.060967,0.008272,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",53098,267953,323449
4,5,AL05,0.346103,0.635919,0.702694,0.297306,0.183322,0.052956,0.025062,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",122344,224704,351650
5,6,AL06,0.320528,0.662526,0.716493,0.283507,0.192588,0.046781,0.021853,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ...",105388,239984,349125
6,7,AL07,0.582883,0.405147,0.427028,0.572972,0.506518,0.036561,0.016651,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ...",180321,111517,294526
7,1,CA01,0.544906,0.422732,0.708434,0.291566,0.023875,0.191664,0.040873,0.044906,"MULTIPOLYGON (((-121.32264 41.18387, -121.3308...",118090,199649,326893


In [11]:
# Rename columns to standard analytical schema
gdf_clean = merged_gdf.rename(columns={
    'id': 'District No.',
    'DemPct': 'Harris New',
    'RepPct': 'Trump New',
    'Margin': 'Partisan Index',
    'Harris': 'Harris 24 Raw',
    'Trump 24': 'Trump 24 Raw',
    'Total': 'Total 24 Raw'
})

# Calculate two-party presidential vote shares and margin shift
gdf_clean['Harris 24'] = gdf_clean['Harris 24 Raw'] / gdf_clean['Total 24 Raw']
gdf_clean['Trump 24'] = gdf_clean['Trump 24 Raw'] / gdf_clean['Total 24 Raw']

gdf_clean['Margin 24'] = gdf_clean['Harris 24'] - gdf_clean['Trump 24']
gdf_clean['Margin New'] = gdf_clean['Harris New'] - gdf_clean['Trump New']
gdf_clean['Margin Shift'] = gdf_clean['Margin New'] - gdf_clean['Margin 24']

gdf_clean.head()

,District No.,District,Harris New,Trump New,WhitePct,MinorityPct,BlackPct,HispanicPct,AsianPct,Partisan Index,geometry,Harris 24 Raw,Trump 24 Raw,Total 24 Raw,Harris 24,Trump 24,Margin 24,Margin New,Margin Shift
0,1,AL01,0.314327,0.673261,0.670796,0.329204,0.246268,0.032630,0.018676,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",73003,257060,332700,0.219426,0.772648,-0.553222,-0.358934,0.194288
1,2,AL02,0.422827,0.565831,0.524975,0.475025,0.399287,0.037441,0.022071,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",155603,131721,290033,0.536501,0.454159,0.082342,-0.143004,-0.225346
2,3,AL03,0.261882,0.727057,0.717585,0.282415,0.206950,0.031171,0.021240,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",82654,229676,314869,0.262503,0.729434,-0.466931,-0.465175,0.001756
3,4,AL04,0.163376,0.826469,0.828291,0.171709,0.072246,0.060967,0.008272,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",53098,267953,323449,0.164162,0.828424,-0.664262,-0.663093,0.001169
4,5,AL05,0.346103,0.635919,0.702694,0.297306,0.183322,0.052956,0.025062,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",122344,224704,351650,0.347914,0.638999,-0.291085,-0.289816,0.001269


In [12]:
# Ingest re-tabulated 2020 presidential returns for redrawn maps
pres20_dir = os.path.join(data_dir, "pres2020")
csv_2020_files = glob.glob(os.path.join(pres20_dir, "*2020.csv"))

pres_2020_records = []

for filepath in csv_2020_files:
    state_abbr = os.path.basename(filepath)[:2].upper()
    df_st = pd.read_csv(filepath)
    
    dist_col = [c for c in df_st.columns if 'district' in c.lower() or c.lower() == 'id'][0]
    dem_col = [c for c in df_st.columns if any(k in c.lower() for k in ['dem', 'biden', 'd_'])][0]
    gop_col = [c for c in df_st.columns if any(k in c.lower() for k in ['rep', 'gop', 'trump', 'r_'])][0]
    
    for _, row in df_st.iterrows():
        raw_dist = str(row[dist_col]).strip()
        dist_num = ''.join(filter(str.isdigit, raw_dist))
        district_id = f"{state_abbr}{int(dist_num):02d}" if dist_num else f"{state_abbr}01"
            
        dem_val = float(row[dem_col])
        gop_val = float(row[gop_col])
        
        if dem_val > 1.0 or gop_val > 1.0:
            total_2party = dem_val + gop_val
            margin_2020 = (dem_val - gop_val) / total_2party if total_2party > 0 else 0.0
            dem_pct_2020 = dem_val / total_2party if total_2party > 0 else 0.0
        else:
            margin_2020 = dem_val - gop_val
            dem_pct_2020 = dem_val

        pres_2020_records.append({
            "District": district_id,
            "Biden 20": dem_pct_2020,
            "Trump 20": 1.0 - dem_pct_2020,
            "Margin 20": margin_2020
        })

df_pres_2020 = pd.DataFrame(pres_2020_records)
gdf_clean = gdf_clean.merge(df_pres_2020[['District', 'Margin 20']], on='District', how='left')

# Fall back to 2024 margin if earlier election data is unavailable
gdf_clean['Margin 20'] = gdf_clean['Margin 20'].fillna(gdf_clean['Margin New'])
gdf_clean.head()

,District No.,District,Harris New,Trump New,WhitePct,MinorityPct,BlackPct,HispanicPct,AsianPct,Partisan Index,geometry,Harris 24 Raw,Trump 24 Raw,Total 24 Raw,Harris 24,Trump 24,Margin 24,Margin New,Margin Shift,Margin 20
0,1,AL01,0.314327,0.673261,0.670796,0.329204,0.246268,0.032630,0.018676,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",73003,257060,332700,0.219426,0.772648,-0.553222,-0.358934,0.194288,-0.239355
1,2,AL02,0.422827,0.565831,0.524975,0.475025,0.399287,0.037441,0.022071,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",155603,131721,290033,0.536501,0.454159,0.082342,-0.143004,-0.225346,-0.051075
2,3,AL03,0.261882,0.727057,0.717585,0.282415,0.206950,0.031171,0.021240,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",82654,229676,314869,0.262503,0.729434,-0.466931,-0.465175,0.001756,-0.356310
3,4,AL04,0.163376,0.826469,0.828291,0.171709,0.072246,0.060967,0.008272,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",53098,267953,323449,0.164162,0.828424,-0.664262,-0.663093,0.001169,-0.557366
4,5,AL05,0.346103,0.635919,0.702694,0.297306,0.183322,0.052956,0.025062,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",122344,224704,351650,0.347914,0.638999,-0.291085,-0.289816,0.001269,-0.228716


In [13]:
# Load 2024 House summary attributes
house_path = os.path.join(data_dir, "pres2024", "house_2024_summary.csv")

if os.path.exists(house_path):
    house_df = pd.read_csv(house_path)
    
    state_lookup = {
        'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
        'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
        'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
        'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
        'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
        'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
        'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
        'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
        'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
        'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
    }

    if 'District_Hover' not in house_df.columns or 'District_Name' not in house_df.columns:
        def make_names(row):
            code = str(row['District']).strip().upper()
            st_abbr = code[:2]
            st_name = state_lookup.get(st_abbr, st_abbr)
            try:
                num = int(code[2:])
            except ValueError:
                num = 1
                
            if num == 0:
                short = f"{st_name} At-Large"
                full = f"{st_name} At-Large District"
            else:
                j, k = num % 10, num % 100
                suf = "st" if (j == 1 and k != 11) else ("nd" if (j == 2 and k != 12) else ("rd" if (j == 3 and k != 13) else "th"))
                short = f"{st_name} {num}{suf}"
                full = f"{st_name} {num}{suf} District"

            rtype = str(row.get('House_Race_Type', '')).lower()
            is_non_std = (rtype != '') and not ('standard' in rtype and 'non-standard' not in rtype)
            star = " *" if is_non_std else ""
            return pd.Series([short + star, full + star], index=['District_Hover', 'District_Name'])

        house_df[['District_Hover', 'District_Name']] = house_df.apply(make_names, axis=1)

    merge_cols = [c for c in [
        'District', 'House_Winner', 'Current_Rep', 'House_Party', 'House_MOV_Raw', 
        'House_Race_Type', 'Effective_House_Margin', 'District_Hover', 'District_Name'
    ] if c in house_df.columns]
    
    gdf_clean = gdf_clean.merge(house_df[merge_cols], on='District', how='left')

# National presidential vote
NATL_MARGIN_2020 = 0.0445
NATL_MARGIN_2024 = -0.0147

# Partisan Lean Index: 70% 2024 + 30% 2020, normalized against national vote
gdf_clean['Partisan Index'] = (
    0.70 * (gdf_clean['Margin New'] - NATL_MARGIN_2024) +
    0.30 * (gdf_clean['Margin 20'] - NATL_MARGIN_2020)
)

# Format partisan lean index
def format_lean_number(val):
    if pd.isna(val) or abs(val) < 0.0005:
        return "EVEN"
    num = abs(val) * 100
    return f"D +{num:.1f}" if val > 0 else f"R +{num:.1f}"
    
gdf_clean['Partisan_Lean_Index'] = gdf_clean['Partisan Index'].apply(format_lean_number)
    
gdf_clean['Partisan_Lean_Index'] = gdf_clean['Partisan Index'].apply(format_lean_number)
gdf_clean.head()

,District No.,District,Harris New,Trump New,WhitePct,MinorityPct,BlackPct,HispanicPct,AsianPct,Partisan Index,...,Margin 20,House_Winner,Current_Rep,House_Party,House_MOV_Raw,House_Race_Type,Effective_House_Margin,District_Hover,District_Name,Partisan_Lean_Index
0,1,AL01,0.314327,0.673261,0.670796,0.329204,0.246268,0.032630,0.018676,-0.326120,...,-0.239355,Barry Moore,Barry Moore,REP,0.569538,Standard General Contest,-0.569538,Alabama 1st,Alabama 1st District,R +32.6
1,2,AL02,0.422827,0.565831,0.524975,0.475025,0.399287,0.037441,0.022071,-0.118485,...,-0.051075,Shomari Figures,Shomari Figures,DEM,0.091990,Standard General Contest,0.091990,Alabama 2nd,Alabama 2nd District,R +11.8
2,3,AL03,0.261882,0.727057,0.717585,0.282415,0.206950,0.031171,0.021240,-0.435576,...,-0.356310,Mike Rogers,Mike Rogers,REP,0.958556,Unopposed / Third-Party Challenger,NaN,Alabama 3rd *,Alabama 3rd District *,R +43.6
3,4,AL04,0.163376,0.826469,0.828291,0.171709,0.072246,0.060967,0.008272,-0.634435,...,-0.557366,Robert B. Aderholt,Robert B. Aderholt,REP,0.975715,Unopposed / Third-Party Challenger,NaN,Alabama 4th *,Alabama 4th District *,R +63.4
4,5,AL05,0.346103,0.635919,0.702694,0.297306,0.183322,0.052956,0.025062,-0.274546,...,-0.228716,Dale W. Strong,Dale W. Strong,REP,0.907869,Unopposed / Third-Party Challenger,NaN,Alabama 5th *,Alabama 5th District *,R +27.5


In [14]:
# Flag targeted flip seats
gdf_clean["Targeted"] = (gdf_clean["Margin New"] * gdf_clean["Margin 24"]) < 0
gdf_clean["Targeted"] = gdf_clean["Targeted"].astype(bool)

# Specific manual overrides for targeted seats
targeted_removals = ["CA09", "CA13", "FL13"]
gdf_clean.loc[gdf_clean["District"].isin(targeted_removals), "Targeted"] = False

targeted_additions = ["NC01", "OH09", "TX28", "TX34", "CA22"]
gdf_clean.loc[gdf_clean["District"].isin(targeted_additions), "Targeted"] = True

gdf_clean["Target_Party"] = None
mask_targeted = gdf_clean["Targeted"] == True
gdf_clean.loc[mask_targeted, "Target_Party"] = gdf_clean.loc[mask_targeted, "Margin Shift"].apply(
    lambda s: "DEM" if s > 0 else "GOP"
)
gdf_clean.loc[gdf_clean["District"] == "CA22", "Target_Party"] = "DEM"

# =========================================================================
# SYSTEMATIC DILUTED / DONOR SEAT FLAG (FILTERING OUT NOISE)
# =========================================================================
# Rules:
# 1. Must shift > 0.5% towards opposition:
#    - For CA (Dem map): Margin Shift < -0.005 (shifting rightward)
#    - For GOP states: Margin Shift > +0.005 (shifting leftward)
# 2. Must be genuinely exposed: Absolute Partisan Index < 20.0% (|Partisan Index| < 0.20)
# 3. Exclude targeted flip seats
gdf_clean["State_Postal"] = gdf_clean["District"].str[:2].str.upper()

cond_ca_diluted = (
    (gdf_clean["State_Postal"] == "CA") & 
    (gdf_clean["Margin Shift"] < -0.005) & 
    (gdf_clean["Partisan Index"].abs() < 0.20)
)

cond_gop_diluted = (
    (gdf_clean["State_Postal"] != "CA") & 
    (gdf_clean["Margin Shift"] > 0.005) & 
    (gdf_clean["Partisan Index"].abs() < 0.20)
)

gdf_clean["Diluted"] = cond_ca_diluted | cond_gop_diluted
gdf_clean.loc[gdf_clean["Targeted"] == True, "Diluted"] = False

# Open seat / Retirement tracking
retiring_districts = ["CA11", "CA26", "CA48", "FL11", "FL16", "FL24", "NC11", "TN09", "TX10", "TX22", "TX33", "TX37"]
gdf_clean["Incumbent_Running"] = ~gdf_clean["District"].isin(retiring_districts)

reordered_columns = [
    "District No.", "District", "District_Hover", "District_Name", "WhitePct", 
    "MinorityPct", "BlackPct", "HispanicPct", "AsianPct", "Partisan_Lean_Index", 
    "House_Winner", "Current_Rep", "House_Party", "House_MOV_Raw", "House_Race_Type", 
    "Harris New", "Trump New", "Margin New", "Margin Shift", "Partisan Index", 
    "Harris 24", "Trump 24", "Margin 24", "Targeted", "Diluted", "Incumbent_Running", 
    "Target_Party", "geometry"
]

export_columns = [col for col in reordered_columns if col in gdf_clean.columns]
gdf_clean = gdf_clean[export_columns]

export_path = "docs/map2026.geojson" if os.path.exists("docs") else "../docs/map2026.geojson"
gdf_clean.to_file(export_path, driver="GeoJSON")
print(f"Exported clean GeoJSON with filtered Diluted flags to {export_path}")

Exported clean GeoJSON with filtered Diluted flags to ../docs/map2026.geojson


In [15]:
# # Generate clean, accurately proportioned SVG path strings from the Census shapefile
# import json
# import matplotlib.pyplot as plt
# from shapely.affinity import translate, scale
# from matplotlib.patches import Circle

# data_dir = "data" if os.path.exists("data") else "../data"
# shp_matches = glob.glob(os.path.join(data_dir, "cb_*", "*.shp"))
# if not shp_matches:
#     shp_matches = glob.glob("data/cb_*/*.shp") or glob.glob("../data/cb_*/*.shp")

# states_raw = gpd.read_file(shp_matches[0]).to_crs("EPSG:9311")

# image_output_dir = os.path.join("docs", "images")
# if not os.path.exists("docs") and os.path.exists("../docs"):
#     image_output_dir = os.path.join("../docs", "images")
# os.makedirs(image_output_dir, exist_ok=True)

# # 2. Export party-colored state shapes for Chart 2 (CA = Blue, others = Red)
# state_parties = {
#     "CA": "#2563eb",  # Democratic (Blue)
#     "TX": "#dc2626",  # Republican (Red)
#     "FL": "#dc2626",  # Republican (Red)
#     "OH": "#dc2626",  # Republican (Red)
#     "NC": "#dc2626",  # Republican (Red)
#     "TN": "#dc2626",  # Republican (Red)
#     "AL": "#dc2626",  # Republican (Red)
#     "LA": "#dc2626"   # Republican (Red)
# }

# for abbr, color in state_parties.items():
#     match_row = states_raw[states_raw["STUSPS"] == abbr]
#     if match_row.empty:
#         continue
    
#     fig, ax = plt.subplots(figsize=(3, 3))
#     match_row.plot(ax=ax, color=color, edgecolor="#0f172a", linewidth=2.0)
#     ax.set_axis_off()
#     plt.tight_layout()
    
#     out_img_path = os.path.join(image_output_dir, f"{abbr}_shape.png")
#     plt.savefig(out_img_path, dpi=300, transparent=True, bbox_inches='tight')
#     plt.close(fig)
#     print(f"Exported party-colored state shape: {out_img_path}")

In [16]:
# Generate favicon
# from matplotlib.patches import Circle
# from PIL import Image

# data_dir = "data" if os.path.exists("data") else "../data"
# shp_matches = glob.glob(os.path.join(data_dir, "cb_*", "*.shp"))
# if not shp_matches:
#     shp_matches = glob.glob("data/cb_*/*.shp") or glob.glob("../data/cb_*/*.shp")

# states_raw = gpd.read_file(shp_matches[0]).to_crs("EPSG:9311")

# # 2. Setup output directories
# docs_dir = "docs" if os.path.exists("docs") else "../docs"
# image_output_dir = os.path.join(docs_dir, "images")
# os.makedirs(image_output_dir, exist_ok=True)
# os.makedirs(docs_dir, exist_ok=True)

# # 3. Grab original Texas geometry and its fixed bounds
# tx_geom = states_raw.loc[states_raw["STUSPS"] == "TX", "geometry"].values[0]
# minx, miny, maxx, maxy = tx_geom.bounds

# # Scale down by 0.8 around its centroid for balanced padding inside the circle
# tx_scaled = scale(tx_geom, xfact=0.8, yfact=0.8, origin='centroid')
# tx_row = gpd.GeoDataFrame(geometry=[tx_scaled], crs=states_raw.crs)

# # 4. Plot inside a fixed-limit canvas so scaling shrinks the visual shape cleanly
# fig, ax = plt.subplots(figsize=(2, 2), subplot_kw={'aspect': 'equal'})

# # Lock data limits to the unscaled Texas bounding box to prevent auto-zooming
# ax.set_xlim(minx, maxx)
# ax.set_ylim(miny, maxy)

# # Draw circular badge
# circ = Circle((0.5, 0.5), 0.48, transform=ax.transAxes, facecolor="#ffffff", edgecolor="#2563eb", linewidth=5)
# ax.add_patch(circ)

# # Plot scaled Texas shape
# tx_row.plot(ax=ax, color="#dc2626", edgecolor="#0f172a", linewidth=1.2)
# ax.set_axis_off()
# plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

# # 5. Save to a temporary PNG inside docs/images/
# temp_png = os.path.join(image_output_dir, "temp_favicon.png")
# plt.savefig(temp_png, dpi=300, transparent=True, bbox_inches='tight', pad_inches=0)
# plt.close(fig)

# # 6. Convert and export as a true multi-resolution .ico file directly to docs/ root
# img = Image.open(temp_png).convert("RGBA")
# root_ico_path = os.path.join(docs_dir, "favicon.ico")

# img.save(root_ico_path, format="ICO", sizes=[(32, 32), (64, 64)])

# print("Successfully generated and exported Favicon")